In [ ]:
import torch, platform, subprocess, os

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(p.total_memory / 1024**3, 2))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

print("\n--- nvidia-smi ---")
subprocess.run(["nvidia-smi"])

Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56
Compute capability: (7, 5)
BF16 supported: True

--- nvidia-smi ---


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

root = Path("/content/drive/MyDrive/EvoVariantTR")
root.mkdir(parents=True, exist_ok=True)

for name in [
    "reference",
    "datasets",
    "checkpoints",
    "model_cache",
    "runs",
    "hpo",
    "logs",
    "state",
    "exports",
]:
    (root / name).mkdir(exist_ok=True)

test_file = root / "state" / "colab_write_test.txt"
test_file.write_text("EvoVariant Colab persistence OK\n")

print(test_file)
print(test_file.read_text())

/content/drive/MyDrive/EvoVariantTR/state/colab_write_test.txt
EvoVariant Colab persistence OK



In [4]:
!git clone -b research/posthoc-foundation-adaptation \
  https://github.com/UtkarsHMer05/EvoVariant-TR-.git \
  /content/EvoVariant

Cloning into '/content/EvoVariant'...
remote: Enumerating objects: 3645, done.
remote: Counting objects: 100% (3645/3645), done.
remote: Compressing objects: 100% (1249/1249), done.
remote: Total 3645 (delta 2388), reused 3510 (delta 2253), pack-reused 0 (from 0)
Receiving objects: 100% (3645/3645), 15.72 MiB | 12.00 MiB/s, done.
Resolving deltas: 100% (2388/2388), done.


In [5]:
%cd /content/EvoVariant
!git branch --show-current
!git rev-parse HEAD

/content/EvoVariant
research/posthoc-foundation-adaptation
dc9ee30e5b149832d0adb2f5d64707ec095d713b


In [6]:
print("EVOVARIANT_PREREQUISITES_READY")

EVOVARIANT_PREREQUISITES_READY


In [7]:
from pathlib import Path
root = Path('/content/drive/MyDrive/EvoVariantTR')
print('drive_root_exists', root.exists())
for rel in ['reference', 'datasets', 'checkpoints', 'model_cache', 'state', 'EvoVariant_TR_Adaptation_Autonomous.ipynb']:
    p = root / rel
    print(rel, 'exists=', p.exists(), 'is_dir=', p.is_dir(), 'size=', p.stat().st_size if p.is_file() else '')
print('repo_exists', Path('/content/EvoVariant').exists())
print('reference_files', [p.name for p in (root / 'reference').glob('*')][:20])
print('repo_reference_files', [p.name for p in (Path('/content/EvoVariant') / 'data/reference').glob('*')][:20] if (Path('/content/EvoVariant') / 'data/reference').exists() else [])

drive_root_exists True
reference exists= True is_dir= True size= 
datasets exists= True is_dir= True size= 
checkpoints exists= True is_dir= True size= 
model_cache exists= True is_dir= True size= 
state exists= True is_dir= True size= 
EvoVariant_TR_Adaptation_Autonomous.ipynb exists= False is_dir= False size= 
repo_exists True
reference_files []
repo_reference_files []


In [8]:
import os, subprocess
os.chdir('/content/EvoVariant')
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'research/posthoc-foundation-adaptation'], check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print(subprocess.check_output(['git', 'status', '--short', '--branch'], text=True).strip())

dc9ee30e5b149832d0adb2f5d64707ec095d713b
## research/posthoc-foundation-adaptation...origin/research/posthoc-foundation-adaptation


In [9]:
from pathlib import Path
import shutil, subprocess, sys
env = Path('/content/caduceus-env')
if not env.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    uv = shutil.which('uv') or '/usr/local/bin/uv'
    subprocess.run([uv, 'venv', '--python', '3.11', str(env)], check=True)
py = str(env / 'bin' / 'python')
subprocess.run([py, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([py, '-m', 'pip', 'install', '-q', '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.2.0'], check=True)
packages = ['transformers==4.38.1', 'accelerate>=0.27', 'huggingface_hub>=0.24', 'mamba-ssm==1.2.0.post1', 'causal-conv1d==1.2.0.post2', 'pyfaidx==0.8.1.1', 'scikit-learn>=1.3', 'pandas>=2.0', 'optuna>=3.6', 'peft>=0.10', 'matplotlib>=3.7', 'seaborn>=0.13', 'safetensors', 'PyYAML', 'tqdm', 'ninja']
subprocess.run([py, '-m', 'pip', 'install', '-q', '--no-build-isolation', *packages], check=True)
print(subprocess.check_output([py, '-c', 'import torch, transformers, mamba_ssm; print(torch.__version__, transformers.__version__, torch.cuda.is_available(), mamba_ssm.__file__)'], text=True))

CalledProcessError: Command '['/content/caduceus-env/bin/python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip']' returned non-zero exit status 1.

In [ ]:
import sys, subprocess
from pathlib import Path
env = Path('/content/caduceus-env')
py = env / 'bin' / 'python'
print('env_exists', env.exists(), 'python_exists', py.exists())
print('host_python', sys.version)
if py.exists():
    r = subprocess.run([str(py), '-m', 'pip', 'install', '--upgrade', 'pip'], text=True, capture_output=True)
    print('returncode', r.returncode)
    print((r.stdout + '\n' + r.stderr)[-6000:])

env_exists True python_exists True
host_python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
returncode 1

/content/caduceus-env/bin/python: No module named pip



In [ ]:
from pathlib import Path
import shutil, subprocess
env = Path('/content/caduceus-env')
uv = shutil.which('uv') or '/usr/local/bin/uv'
py = str(env / 'bin' / 'python')
commands = [[uv, 'pip', 'install', '--python', py, 'setuptools', 'wheel', 'packaging', 'ninja'], [uv, 'pip', 'install', '--python', py, '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.2.0'], [uv, 'pip', 'install', '--python', py, '--no-build-isolation', 'transformers==4.38.1', 'accelerate>=0.27', 'huggingface_hub>=0.24', 'mamba-ssm==1.2.0.post1', 'causal-conv1d==1.2.0.post2', 'pyfaidx==0.8.1.1', 'scikit-learn>=1.3', 'pandas>=2.0', 'optuna>=3.6', 'peft>=0.10', 'matplotlib>=3.7', 'seaborn>=0.13', 'safetensors', 'PyYAML', 'tqdm']]
for command in commands:
    result = subprocess.run(command, text=True, capture_output=True)
    print('returncode', result.returncode)
    print((result.stdout + '\n' + result.stderr)[-4000:])
    result.check_returncode()
print(subprocess.check_output([py, '-c', 'import torch, transformers, mamba_ssm; print(torch.__version__, transformers.__version__, torch.cuda.is_available(), mamba_ssm.__file__)'], text=True))

returncode 0

Using Python 3.11.16 environment at: /content/caduceus-env
Resolved 4 packages in 102ms
Prepared 4 packages in 35ms
Installed 4 packages in 9ms
 + ninja==1.13.2
 + packaging==26.3
 + setuptools==84.0.0
 + wheel==0.48.0

returncode 0

Using Python 3.11.16 environment at: /content/caduceus-env
Checked 1 package in 2ms

returncode 0

Using Python 3.11.16 environment at: /content/caduceus-env
Resolved 71 packages in 8.83s
Prepared 46 packages in 20.00s
Installed 46 packages in 214ms
 + accelerate==1.15.0
 + alembic==1.20.0
 + causal-conv1d==1.2.0.post2
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + cloudpickle==3.1.2
 + colorlog==6.12.0
 + contourpy==1.3.3
 + cycler==0.12.1
 + einops==0.8.2
 + fonttools==4.65.0
 + greenlet==3.5.6
 + hf-xet==1.6.0
 + huggingface-hub==0.36.2
 + idna==3.20
 + importlib-metadata==9.0.1
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + mako==1.4.1
 + mamba-ssm==1.2.0.post1
 + matplotlib==3.11.2
 + narwhals==2.26.0
 + numpy==2.4.6
 + optuna==5.0.0
 +

In [ ]:
from pathlib import Path
import os, subprocess
root = Path('/content/EvoVariant')
os.chdir(root)
subprocess.run(['git','pull','--ff-only','origin','research/posthoc-foundation-adaptation'], check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print(subprocess.check_output(['git','status','--short','--branch'], text=True).strip())

e7657827d59f5d2dce64725a2e93a9d45419e155
## research/posthoc-foundation-adaptation...origin/research/posthoc-foundation-adaptation


In [ ]:
from pathlib import Path
import subprocess
ref_dir = Path('/content/drive/MyDrive/EvoVariantTR/reference')
ref_dir.mkdir(parents=True, exist_ok=True)
reference = ref_dir / 'Homo_sapiens_assembly38.fasta'
if not reference.exists():
    archive = ref_dir / 'hg38.fa.gz'
    subprocess.run(['wget','-c','https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz','-O',str(archive)], check=True)
    subprocess.run(['gunzip','-f',str(archive)], check=True)
    (ref_dir / 'hg38.fa').replace(reference)
py = '/content/caduceus-env/bin/python'
subprocess.run([py,'-c',f'from pyfaidx import Fasta; Fasta({str(reference)!r})'], check=True)
print(reference, reference.stat().st_size, reference.with_suffix('.fasta.fai').exists())

CalledProcessError: Command '['wget', '-c', 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz', '-O', '/content/drive/MyDrive/EvoVariantTR/reference/hg38.fa.gz']' returned non-zero exit status 4.

In [ ]:
from pathlib import Path
import subprocess
d = Path('/content/drive/MyDrive/EvoVariantTR/reference')
d.mkdir(parents=True, exist_ok=True)
reference = d / 'Homo_sapiens_assembly38.fasta'
archive = d / 'hg38.fa.gz'
url = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz'
subprocess.run(['curl','-L','--retry','3','--fail','--output',str(archive),url], check=True)
subprocess.run(['gunzip','-f',str(archive)], check=True)
(d / 'hg38.fa').replace(reference)
py = '/content/caduceus-env/bin/python'
subprocess.run([py,'-c',f'from pyfaidx import Fasta; Fasta({str(reference)!r})'], check=True)
import socket, shutil, subprocess, sys
from pathlib import Path
for host in ('hgdownload.soe.ucsc.edu', 'huggingface.co', 'pypi.org'):
    try: print(host, socket.gethostbyname_ex(host))
    except Exception as exc: print(host, type(exc).__name__, str(exc))
root = Path('/content/drive/MyDrive/EvoVariantTR')
print('drive_root_exists', root.exists(), 'free_GiB', round(shutil.disk_usage(root).free / 1024**3, 2))
print('reference_files', [(p.name, p.stat().st_size) for p in (root / 'reference').glob('*')])
print('repo_HEAD', subprocess.check_output(['git', '-C', '/content/EvoVariant', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import subprocess
from pathlib import Path
root=Path('/content/drive/MyDrive/EvoVariantTR')
urls=['https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz','https://huggingface.co/api/models/kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16','https://pypi.org/simple/']
print('reference_files',[(p.name,p.stat().st_size) for p in (root/'reference').glob('*')])
results=[subprocess.run(['curl','-sSIL','--max-time','8',url],capture_output=True,text=True) for url in urls]
print([(url,r.returncode,r.stdout.splitlines()[:2],r.stderr[-120:]) for url,r in zip(urls,results)])

reference_files [('hg38.fa.gz', 0)]
[('https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz', 6, [], 'curl: (6) Could not resolve host: hgdownload.soe.ucsc.edu\n'), ('https://huggingface.co/api/models/kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16', 0, ['HTTP/2 200 ', 'content-type: application/json; charset=utf-8'], ''), ('https://pypi.org/simple/', 0, ['HTTP/2 200 ', "content-security-policy: default-src 'none'; sandbox allow-top-navigation"], '')]


In [ ]:
import subprocess
r=subprocess.run(['curl','--resolve','hgdownload.soe.ucsc.edu:443:128.114.119.163','-sSIL','--connect-timeout','5','--max-time','10','https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz'],capture_output=True,text=True)
print('returncode',r.returncode,'headers',r.stdout.splitlines()[:4],'error',r.stderr)

returncode 0 headers ['HTTP/1.1 200 OK', 'Date: Tue, 22 Sep 2026 19:12:41 GMT', 'Server: Apache', 'Last-Modified: Thu, 16 Jan 2014 05:14:12 GMT'] error 


In [ ]:
from pathlib import Path
import subprocess
d=Path('/content/drive/MyDrive/EvoVariantTR/reference')
d.mkdir(parents=True,exist_ok=True)
archive=d/'hg38.fa.gz'
url='https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz'
subprocess.run(['curl','-L','--resolve','hgdownload.soe.ucsc.edu:443:128.114.119.163','--retry','3','--fail','--connect-timeout','20','--max-time','1800','--output',str(archive),url],check=True)
print('archive_bytes',archive.stat().st_size)

archive_bytes 983659424


In [ ]:
import gzip,hashlib,shutil,subprocess
archive=DRIVE_ROOT/'reference'/'hg38.fa.gz'
reference=Path('/content/Homo_sapiens_assembly38.fasta')
digest=hashlib.sha256()
with gzip.open(archive,'rb') as source, reference.open('wb') as target:
    while chunk:=source.read(8*1024*1024):
        target.write(chunk)
        digest.update(chunk)
actual=digest.hexdigest()
expected='93157a161863464c9435062fd67c173fdaf99cb8b32f1455018361387ffa5564'
assert actual==expected,(actual,expected)
subprocess.run([PY,'-c',f'from pyfaidx import Fasta; Fasta({str(reference)!r})'],check=True)
shutil.copy2(reference.with_suffix('.fasta.fai'),DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai')
print('reference',reference,'bytes',reference.stat().st_size,'sha256',actual)
print('drive_archive_bytes',archive.stat().st_size,'fai_bytes',(DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai').stat().st_size)

NameError: name 'DRIVE_ROOT' is not defined

In [ ]:
import gzip,hashlib,shutil,subprocess
DRIVE_ROOT=Path('/content/drive/MyDrive/EvoVariantTR')
archive=DRIVE_ROOT/'reference'/'hg38.fa.gz'
assert archive.stat().st_size==983659424,archive.stat().st_size
reference=Path('/content/Homo_sapiens_assembly38.fasta')
digest=hashlib.sha256()
with gzip.open(archive,'rb') as source, reference.open('wb') as target:
    while chunk:=source.read(8*1024*1024):
        target.write(chunk)
        digest.update(chunk)
actual=digest.hexdigest()
expected='93157a161863464c9435062fd67c173fdaf99cb8b32f1455018361387ffa5564'
assert actual==expected,(actual,expected)
PY='/content/caduceus-env/bin/python'
subprocess.run([PY,'-c',f'from pyfaidx import Fasta; Fasta({str(reference)!r})'],check=True)
shutil.copy2(reference.with_suffix('.fasta.fai'),DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai')
print('reference',reference,'bytes',reference.stat().st_size,'sha256',actual)
print('drive_archive_bytes',archive.stat().st_size,'fai_bytes',(DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai').stat().st_size)

AssertionError: ('5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51', '93157a161863464c9435062fd67c173fdaf99cb8b32f1455018361387ffa5564')

In [ ]:
p=Path('/content/Homo_sapiens_assembly38.fasta')
with p.open('rb') as f: print([f.readline().decode().strip() for _ in range(3)])
print('size',p.stat().st_size,'raw_sha256',actual)

['>chr1', 'NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN', 'NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN']
size 3273481150 raw_sha256 5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51


In [ ]:
import os, subprocess
from pathlib import Path
ROOT=Path('/content/EvoVariant')
PY='/content/caduceus-env/bin/python'
PROGRAM = r'''import hashlib, json, shutil
from pathlib import Path
from pyfaidx import Fasta
from evovariant_tr.adaptation.data import load_formal_rows, resolve_contig
root=Path('/content/EvoVariant')
reference=Path('/content/Homo_sapiens_assembly38.fasta')
archive=Path('/content/drive/MyDrive/EvoVariantTR/reference/hg38.fa.gz')
assert archive.stat().st_size == 983659424, archive.stat().st_size
digest=hashlib.sha256()
with reference.open('rb') as handle:
    for chunk in iter(lambda:handle.read(8*1024*1024),b''):
        digest.update(chunk)
reference_sha256=digest.hexdigest()
assert reference_sha256 == '5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51', reference_sha256
train, validation=load_formal_rows(root)
fasta=Fasta(str(reference),as_raw=True,sequence_always_upper=True)
for row in (*train,*validation):
    contig=resolve_contig(fasta,row.chromosome)
    start=row.position_1based-1
    observed=str(fasta[contig][start:start+len(row.reference)]).upper()
    if observed != row.reference:
        raise RuntimeError(f'REF mismatch {row.normalized_variant_id}: expected {row.reference}, got {observed}')
shutil.copy2(reference.with_suffix('.fasta.fai'),archive.parent/'Homo_sapiens_assembly38.fasta.fai')
print(json.dumps({'status':'REF_ALLELES_PASS','checked':len(train)+len(validation),'train':len(train),'validation':len(validation),'reference_bytes':reference.stat().st_size,'reference_sha256':reference_sha256,'fai_bytes':(archive.parent/'Homo_sapiens_assembly38.fasta.fai').stat().st_size,'locked_rows_loaded':False},indent=2,sort_keys=True))'''
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
result=subprocess.run([PY,'-c',PROGRAM],cwd=ROOT,env=env,text=True,capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr[-4000:])
    result.check_returncode()

{
  "checked": 4000,
  "fai_bytes": 19381,
  "locked_rows_loaded": false,
  "reference_bytes": 3273481150,
  "reference_sha256": "5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51",
  "status": "REF_ALLELES_PASS",
  "train": 3199,
  "validation": 801
}



In [ ]:
import os, subprocess
from pathlib import Path
ROOT=Path('/content/EvoVariant')
result=subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],text=True,capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()
print(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print(subprocess.check_output(['git','-C',str(ROOT),'status','--short','--branch'],text=True).strip())

Updating e765782..f1ccf55
Fast-forward
 .../EvoVariant_TR_Adaptation_Autonomous.ipynb      | 103 ++++-
 pyproject.toml                                     |   6 +
 research/adaptation/BRANCH_HYGIENE.md              |   2 +-
 scripts/adaptation/evaluate_caduceus.py            | 114 ++++-
 scripts/adaptation/hardware_probe.py               |  55 ++-
 scripts/adaptation/run_caduceus_hpo.py             | 491 +++++++++++++++++----
 scripts/adaptation/run_posthoc_analysis.py         |   1 -
 scripts/adaptation/smoke_caduceus.py               |  82 +++-
 scripts/adaptation/train_caduceus.py               | 138 +++++-
 scripts/adaptation/verify_data.py                  |  45 +-
 src/evovariant_tr/adaptation/__init__.py           |   7 +
 src/evovariant_tr/adaptation/calibration.py        |  15 +-
 src/evovariant_tr/adaptation/checkpointing.py      |  19 +-
 src/evovariant_tr/adaptation/data.py               |  48 +-
 src/evovariant_tr/adaptation/folds.py              |   1 -
 src/evovariant_tr

In [ ]:
import os, subprocess
from pathlib import Path
ROOT=Path('/content/EvoVariant')
DRIVE_ROOT=Path('/content/drive/MyDrive/EvoVariantTR')
PY='/content/caduceus-env/bin/python'
STATE=DRIVE_ROOT/'state'/'adaptation_state.json'
REFERENCE=Path('/content/Homo_sapiens_assembly38.fasta')
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
commands=[
    [PY,'scripts/adaptation/hardware_probe.py','--root',str(ROOT),'--output',str(DRIVE_ROOT/'state'/'environment.json'),'--state',str(STATE)],
    [PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--output',str(DRIVE_ROOT/'state'/'manifests_verified.json'),'--state',str(STATE)],
    [PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--reference',str(REFERENCE),'--output',str(DRIVE_ROOT/'state'/'data_ready.json'),'--state',str(STATE)],
]
for command in commands:
    subprocess.run(command,cwd=ROOT,env=env,check=True)

In [ ]:
from pathlib import Path
import json
base=Path('/content/drive/MyDrive/EvoVariantTR/state')
for name in ['environment.json','manifests_verified.json','data_ready.json','adaptation_state.json']:
 p=base/name
 print(name, 'exists=',p.exists(), 'bytes=',p.stat().st_size if p.exists() else None)
 if p.exists() and p.stat().st_size < 20000: print(p.read_text()[:2000])


environment.json exists= True bytes= 774
{
  "compute_capability": [
    7,
    5
  ],
  "cuda_available": true,
  "cuda_runtime": "12.1",
  "gpu": "Tesla T4",
  "model_revisions": {
    "caduceus": "b0477522ac5d044ad03578aa724ec8e4bdbd405b",
    "nucleotide_transformer": "06615c1660c892fc199840c18123f8385b3542a8"
  },
  "nvidia_smi": "Tesla T4, 15360 MiB, 580.82.07",
  "packages": {
    "causal-conv1d": "1.2.0.post2",
    "mamba-ssm": "1.2.0.post1",
    "numpy": "2.4.6",
    "optuna": "5.0.0",
    "pandas": "3.0.6",
    "peft": "0.21.0",
    "scikit-learn": "1.9.1",
    "torch": "2.2.0+cu121",
    "transformers": "4.38.1"
  },
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.11.16 (main, Sep  1 2026, 14:18:37) [Clang 22.1.3 ]",
  "torch": "2.2.0+cu121",
  "vram_bytes": 15637086208
}

manifests_verified.json exists= True bytes= 760
{
  "formal_record_count": 4000,
  "locked_id_overlap": 0,
  "locked_manifest_sha256": "9f9e052d21f4a6a32f595cb20f48cb81e033c0481942820d

In [ ]:
from pathlib import Path
import os, subprocess
ROOT=Path('/content/EvoVariant')
DRIVE=Path('/content/drive/MyDrive/EvoVariantTR')
PY='/content/caduceus-env/bin/python'
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
command=[PY,'scripts/adaptation/smoke_caduceus.py','--root',str(ROOT),'--device','cuda','--cache-dir',str(DRIVE/'model_cache'),'--output',str(DRIVE/'runs'/'caduceus_smoke.json'),'--checkpoint',str(DRIVE/'checkpoints'/'caduceus_smoke.pt'),'--state',str(DRIVE/'state'/'adaptation_state.json')]
print('Starting pinned Caduceus smoke test on the connected T4')
result=subprocess.run(command,cwd=ROOT,env=env,text=True,capture_output=True)
print(result.stdout[-10000:])
if result.returncode:
 print(result.stderr[-10000:]); result.check_returncode()


Starting pinned Caduceus smoke test on the connected T4


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/content/EvoVariant/scripts/adaptation/smoke_caduceus.py", line 156, in <module>
    main()
  File "/content/EvoVariant/scripts/adaptation/smoke_caduceus.py", line 37, in main
    import torch
  File "/content/caduceus-env/lib/python3.11/site-packages/torch/__init__.py", line 1471, in <module>
    from .functional import *  # noqa: F403
  File "/content/caduceus-env/lib/python3.11/site-packages/torch/functional.py", line 9, in <module>
    import torch.nn.fun

CalledProcessError: Command '['/content/caduceus-env/bin/python', 'scripts/adaptation/smoke_caduceus.py', '--root', '/content/EvoVariant', '--device', 'cuda', '--cache-dir', '/content/drive/MyDrive/EvoVariantTR/model_cache', '--output', '/content/drive/MyDrive/EvoVariantTR/runs/caduceus_smoke.json', '--checkpoint', '/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_smoke.pt', '--state', '/content/drive/MyDrive/EvoVariantTR/state/adaptation_state.json']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import subprocess
ROOT=Path('/content/EvoVariant'); PY='/content/caduceus-env/bin/python'; UV='/usr/local/bin/uv'
r=subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],text=True,capture_output=True); print(r.stdout+r.stderr); r.check_returncode()
r=subprocess.run([UV,'pip','install','--python',PY,'numpy==1.26.4'],text=True,capture_output=True); print(r.stdout+r.stderr); r.check_returncode()
r=subprocess.run([PY,'-c','import numpy,torch; print("numpy",numpy.__version__,"torch",torch.__version__,"cuda",torch.cuda.is_available())'],text=True,capture_output=True); print(r.stdout+r.stderr); r.check_returncode()
print(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


Updating f1ccf55..1d4fe87
Fast-forward
 requirements-adaptation.txt               | 2 +-
 src/evovariant_tr/adaptation/models.py    | 7 ++++---
 tests/adaptation/test_research_helpers.py | 8 ++++++++
 3 files changed, 13 insertions(+), 4 deletions(-)
From https://github.com/UtkarsHMer05/EvoVariant-TR-
 * branch            research/posthoc-foundation-adaptation -> FETCH_HEAD
   f1ccf55..1d4fe87  research/posthoc-foundation-adaptation -> origin/research/posthoc-foundation-adaptation

Using Python 3.11.16 environment at: /content/caduceus-env
Resolved 1 package in 344ms
Prepared 1 package in 758ms
Uninstalled 1 package in 29ms
Installed 1 package in 25ms
 - numpy==2.4.6
 + numpy==1.26.4

numpy 1.26.4 torch 2.2.0+cu121 cuda True

1d4fe87e75e513e763ef7ccbc7de2f01ee919b8e


In [ ]:
from pathlib import Path
import os, subprocess
ROOT=Path('/content/EvoVariant'); DRIVE=Path('/content/drive/MyDrive/EvoVariantTR'); PY='/content/caduceus-env/bin/python'
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
for command in [
 [PY,'scripts/adaptation/hardware_probe.py','--root',str(ROOT),'--output',str(DRIVE/'state'/'environment.json'),'--state',str(DRIVE/'state'/'adaptation_state.json')],
 [PY,'scripts/adaptation/smoke_caduceus.py','--root',str(ROOT),'--device','cuda','--cache-dir',str(DRIVE/'model_cache'),'--output',str(DRIVE/'runs'/'caduceus_smoke.json'),'--checkpoint',str(DRIVE/'checkpoints'/'caduceus_smoke.pt'),'--state',str(DRIVE/'state'/'adaptation_state.json')]
]:
 r=subprocess.run(command,cwd=ROOT,env=env,text=True,capture_output=True); print(r.stdout); print(r.stderr[-4000:] if r.returncode else ''); r.check_returncode()


{
  "compute_capability": [
    7,
    5
  ],
  "cuda_available": true,
  "cuda_runtime": "12.1",
  "gpu": "Tesla T4",
  "model_revisions": {
    "caduceus": "b0477522ac5d044ad03578aa724ec8e4bdbd405b",
    "nucleotide_transformer": "06615c1660c892fc199840c18123f8385b3542a8"
  },
  "nvidia_smi": "Tesla T4, 15360 MiB, 580.82.07",
  "packages": {
    "causal-conv1d": "1.2.0.post2",
    "mamba-ssm": "1.2.0.post1",
    "numpy": "1.26.4",
    "optuna": "5.0.0",
    "pandas": "3.0.6",
    "peft": "0.21.0",
    "scikit-learn": "1.9.1",
    "torch": "2.2.0+cu121",
    "transformers": "4.38.1"
  },
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.11.16 (main, Sep  1 2026, 14:18:37) [Clang 22.1.3 ]",
  "torch": "2.2.0+cu121",
  "vram_bytes": 15637086208
}



/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you wa

CalledProcessError: Command '['/content/caduceus-env/bin/python', 'scripts/adaptation/smoke_caduceus.py', '--root', '/content/EvoVariant', '--device', 'cuda', '--cache-dir', '/content/drive/MyDrive/EvoVariantTR/model_cache', '--output', '/content/drive/MyDrive/EvoVariantTR/runs/caduceus_smoke.json', '--checkpoint', '/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_smoke.pt', '--state', '/content/drive/MyDrive/EvoVariantTR/state/adaptation_state.json']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import os, subprocess
ROOT=Path('/content/EvoVariant'); DRIVE=Path('/content/drive/MyDrive/EvoVariantTR'); PY='/content/caduceus-env/bin/python'
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
r=subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],text=True,capture_output=True); print(r.stdout+r.stderr); r.check_returncode()
command=[PY,'scripts/adaptation/smoke_caduceus.py','--root',str(ROOT),'--device','cuda','--cache-dir',str(DRIVE/'model_cache'),'--output',str(DRIVE/'runs'/'caduceus_smoke.json'),'--checkpoint',str(DRIVE/'checkpoints'/'caduceus_smoke.pt'),'--state',str(DRIVE/'state'/'adaptation_state.json')]
r=subprocess.run(command,cwd=ROOT,env=env,text=True,capture_output=True); print(r.stdout); print(r.stderr[-5000:] if r.returncode else ''); r.check_returncode()


Updating 1d4fe87..da23f5d
Fast-forward
 src/evovariant_tr/adaptation/models.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
From https://github.com/UtkarsHMer05/EvoVariant-TR-
 * branch            research/posthoc-foundation-adaptation -> FETCH_HEAD
   1d4fe87..da23f5d  research/posthoc-foundation-adaptation -> origin/research/posthoc-foundation-adaptation

{
  "checkpoint": "/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_smoke.pt",
  "checkpoint_reload_prediction_compatible": true,
  "checkpoint_sha256": "da541bb25b2cc556d5089357939c0dd3a1589aa7c604b1242ec6da71604c9042",
  "device": "cuda",
  "hidden_size": 256,
  "logit_finite": true,
  "loss_finite": true,
  "model_id": "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16",
  "peak_vram_bytes": 452699136,
  "revision": "b0477522ac5d044ad03578aa724ec8e4bdbd405b",
  "runtime_seconds": 10.726524306999636,
  "sequence_length": 128,
  "status": "PASS",
  "total_parameters": 7728385,
  "trainable_parameters": 

In [ ]:
from pathlib import Path
import os, subprocess
ROOT=Path('/content/EvoVariant'); DRIVE=Path('/content/drive/MyDrive/EvoVariantTR'); PY='/content/caduceus-env/bin/python'
subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],check=True)
env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src')
log_path=DRIVE/'logs'/'caduceus_frozen_head.log'; log_path.parent.mkdir(parents=True,exist_ok=True)
command=[PY,'scripts/adaptation/train_caduceus.py','--root',str(ROOT),'--reference','/content/Homo_sapiens_assembly38.fasta','--cache-dir',str(DRIVE/'model_cache'),'--output-dir',str(DRIVE/'checkpoints'/'caduceus_frozen_head'),'--state',str(DRIVE/'state'/'adaptation_state.json'),'--device','cuda','--stage','frozen_head_only','--epochs','3','--batch-size','1','--effective-batch-size','16','--dropout','0.1','--max-length','8192','--learning-rate','2e-5','--weight-decay','0.01','--seed','42']
log_handle=log_path.open('w')
train_process=subprocess.Popen(command,cwd=ROOT,env=env,stdout=log_handle,stderr=subprocess.STDOUT,text=True,start_new_session=True)
print('TRAIN_ONLY frozen-head baseline started; pid=',train_process.pid,'log=',log_path,'HEAD=',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


TRAIN_ONLY frozen-head baseline started; pid= 48223 log= /content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log HEAD= 34f65a3d8d0125be4cf00b78f19ce115cdc9feae


In [ ]:
from pathlib import Path
print('process_exit_code=',train_process.poll())
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
print('log_bytes=',log_path.stat().st_size if log_path.exists() else None)
print(log_path.read_text()[-2000:] if log_path.exists() else '')


process_exit_code= None
log_bytes= 294
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



In [ ]:
import subprocess, time
from pathlib import Path
time.sleep(15)
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
ckpt_dir=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit_code=',train_process.poll(),'log_bytes=',log_path.stat().st_size if log_path.exists() else None)
print(log_path.read_text()[-3000:] if log_path.exists() else '')
print('artifacts=',[(p.name,p.stat().st_size) for p in ckpt_dir.glob('*')] if ckpt_dir.exists() else [])
r=subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True); print('gpu=',r.stdout.strip())


exit_code= None log_bytes= 294
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

artifacts= []
gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
import os, subprocess
from pathlib import Path
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log'); ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('elapsed_process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip(),'exit=',train_process.poll())
print('log_bytes=',log_path.stat().st_size if log_path.exists() else None,'files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')] if ckpt.exists() else [])
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())


elapsed_process= 02:29 98.4 Rsl exit= None
log_bytes= 294 files= []
84 %, 789 MiB, 15360 MiB


In [ ]:
import subprocess
from pathlib import Path
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log'); ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('elapsed_process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip(),'exit=',train_process.poll())
print('log_bytes=',log_path.stat().st_size if log_path.exists() else None,'files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')] if ckpt.exists() else [])
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())


elapsed_process= 03:50 98.4 Rsl exit= None
log_bytes= 294 files= []
88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'elapsed=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('log_bytes=',log_path.stat().st_size if log_path.exists() else None,'artifacts=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')] if ckpt.exists() else [])
print(log_path.read_text()[-5000:] if log_path.exists() else '')
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None elapsed= 06:29 98.6 Rsl
log_bytes= 294 artifacts= []
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

85 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess, json
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])
print('latest=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('log_tail=',log_path.read_text()[-3000:])
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 10:40 98.7 Rsl
files= []
latest= pending
log_tail= /content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

gpu= 82 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('artifacts=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 14:51 98.6 Rsl
artifacts= [('latest.pt', 31077754), ('latest.json', 200)]
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 0,
  "loss": 1.1767539545572263,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
log_path=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('artifacts=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 19:39 98.7 Rsl
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 0,
  "loss": 1.1767539545572263,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

artifacts= [('latest.pt', 31077754), ('latest.json', 200)]
gpu= 68 %, 789 MiB, 15360 MiB


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/EvoVariantTR/state/adaptation_state.json')
s=json.loads(p.read_text())
print('latest_stage=',s.get('stage'))
for e in s.get('history',[])[-8:]: print(e['stage'],e.get('timestamp_utc'),e.get('git_head'),e.get('details',{}).get('epoch'),e.get('details',{}).get('loss'))
for path in [Path('/content/drive/MyDrive/EvoVariantTR/state/environment.json'),Path('/content/drive/MyDrive/EvoVariantTR/state/data_ready.json'),Path('/content/drive/MyDrive/EvoVariantTR/runs/caduceus_smoke.json')]:
 print(path.name, path.read_text()[:1000] if path.exists() else 'MISSING')

latest_stage= CADUCEUS_FROZEN_HEAD_ONLY_TRAIN_PROGRESS
ENV_READY 2026-09-22T19:56:41.467799+00:00 f1ccf553135f01385265647d3bf907082caf9e9a None None
MANIFESTS_VERIFIED 2026-09-22T19:56:42.069506+00:00 f1ccf553135f01385265647d3bf907082caf9e9a None None
DATA_READY 2026-09-22T19:56:53.335962+00:00 f1ccf553135f01385265647d3bf907082caf9e9a None None
ENV_READY 2026-09-22T20:03:38.408077+00:00 1d4fe87e75e513e763ef7ccbc7de2f01ee919b8e None None
CADUCEUS_SMOKE_PASS 2026-09-22T20:05:49.370254+00:00 da23f5ded3407a0a840459fd32670f8da88833fc None None
CADUCEUS_FROZEN_HEAD_ONLY_TRAIN_PROGRESS 2026-09-22T20:20:02.538490+00:00 34f65a3d8d0125be4cf00b78f19ce115cdc9feae 0 1.1767539545572263
environment.json {
  "compute_capability": [
    7,
    5
  ],
  "cuda_available": true,
  "cuda_runtime": "12.1",
  "gpu": "Tesla T4",
  "model_revisions": {
    "caduceus": "b0477522ac5d044ad03578aa724ec8e4bdbd405b",
    "nucleotide_transformer": "06615c1660c892fc199840c18123f8385b3542a8"
  },
  "nvidia_smi": "Tesla

In [ ]:
import subprocess
from pathlib import Path
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('latest=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 24:19 98.7 Rsl
latest= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 1,
  "loss": 1.1581548454985837,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

files= [('latest.pt', 31077754), ('latest.json', 200)]
gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 26:31 98.7 Rsl
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 1,
  "loss": 1.1581548454985837,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('run_report=',(ckpt/'run.json').exists())
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 28:25 98.7 Rsl
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 1,
  "loss": 1.1581548454985837,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

run_report= False
gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 30:02 98.7 Rsl
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 1,
  "loss": 1.1581548454985837,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

files= [('latest.pt', 31077754), ('latest.json', 200)]
gpu= 88 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('run_report=',(ckpt/'run.json').exists())
print('gpu=',subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())

exit= None process= 31:46 98.7 Rsl
progress= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 1,
  "loss": 1.1581548454985837,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

run_report= False
gpu= 84 %, 789 MiB, 15360 MiB


In [ ]:
from pathlib import Path
import subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
print('exit=',train_process.poll(),'process=',subprocess.check_output(['ps','-p',str(train_process.pid),'-o','etime=,pcpu=,stat='],text=True).strip())
print('progress=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'pending')
print('run_report_exists=',(ckpt/'run.json').exists())
print('files=',[(p.name,p.stat().st_size) for p in ckpt.glob('*')])

CalledProcessError: Command '['ps', '-p', '48223', '-o', 'etime=,pcpu=,stat=']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import hashlib,json,subprocess
ckpt=Path('/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head')
log=Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_frozen_head.log')
print('exit_code=',train_process.poll())
print('latest=',(ckpt/'latest.json').read_text() if (ckpt/'latest.json').exists() else 'missing')
print('run=',(ckpt/'run.json').read_text() if (ckpt/'run.json').exists() else 'missing')
print('checkpoint_bytes=',(ckpt/'latest.pt').stat().st_size if (ckpt/'latest.pt').exists() else None)
if (ckpt/'latest.pt').exists():
 h=hashlib.sha256(); f=(ckpt/'latest.pt').open('rb')
 for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
 print('checkpoint_sha256=',h.hexdigest())
print('log_tail=',log.read_text(errors='replace')[-6000:])
s=json.loads(Path('/content/drive/MyDrive/EvoVariantTR/state/adaptation_state.json').read_text())
print('latest_stage=',s.get('stage'))
for e in s['history'][-5:]: print(e['stage'],e.get('details',{}).get('epoch'),e.get('details',{}).get('loss'))

exit_code= 0
latest= {
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "epoch": 2,
  "loss": 1.1497443100928366,
  "stage": "frozen_head_only",
  "status": "TRAIN_ONLY_PROGRESS"
}

run= {
  "checkpoint": "/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head/latest.pt",
  "checkpoint_bytes": 31077754,
  "checkpoint_sha256": "230fc7bef5c12fcb5f5c5a6b31ff4d79d57c244b8a00d70ac54e60aaf4e02213",
  "config": {
    "batch_size": 1,
    "dropout": 0.1,
    "effective_batch_size": 16,
    "epochs": 3,
    "learning_rate": 2e-05,
    "max_length": 8192,
    "reference_sha256": "5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51",
    "seed": 42,
    "stage": "frozen_head_only",
    "weight_decay": 0.01
  },
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "data": {
    "reference_sha256": "5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51",
    "train_manifest_sha256": "32

In [ ]:
import subprocess
ROOT=Path('/content/EvoVariant')
pull=subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],text=True,capture_output=True)
print(pull.stdout+pull.stderr)
pull.check_returncode()
HEAD=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
branch=subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'],text=True).strip()
status=subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain'],text=True).strip()
print('branch=',branch,'HEAD=',HEAD,'worktree=',repr(status))
assert branch=='research/posthoc-foundation-adaptation' and HEAD.startswith('69ad3e6') and not status
print('history persistence present=', 'fold_epoch_history' in (ROOT/'scripts/adaptation/run_caduceus_hpo.py').read_text())

Updating 34f65a3..69ad3e6
Fast-forward
 README.md                                          |  34 ++
 docs/CONTRIBUTION_MAP.md                           |  13 +
 docs/JUDGE_DEMO.md                                 |  19 +
 .../EvoVariant_TR_Adaptation_Autonomous.ipynb      | 319 ++++++++----
 .../EvoVariant_TR_Adaptation_Judge_Demo.ipynb      | 546 ++++++++++++++++++++-
 research/adaptation/DECISIONS.md                   |   9 +-
 research/adaptation/EXPERIMENT_LEDGER.md           |  49 +-
 research/adaptation/STATE.md                       |  99 ++--
 .../reports/adaptation/FINAL_ADAPTATION_REPORT.md  |  82 ++++
 scripts/adaptation/run_caduceus_hpo.py             |  20 +-
 scripts/adaptation/run_posthoc_analysis.py         |  92 +++-
 src/evovariant_tr/adaptation/calibration.py        | 182 ++++++-
 src/evovariant_tr/adaptation/statistics.py         | 105 +++-
 tests/adaptation/test_research_helpers.py          |  70 +++
 14 files changed, 1443 insertions(+), 196 deletions(-)
 create mo

In [ ]:
from pathlib import Path
import json, os, subprocess, time
hpo_root = Path('/content/EvoVariant')
hpo_drive = Path('/content/drive/MyDrive/EvoVariantTR')
hpo_out = hpo_drive / 'hpo/caduceus_hpo.json'
hpo_db = hpo_out.with_suffix('.sqlite3')
hpo_lock = hpo_out.with_name('selection_closed.json')
hpo_state = hpo_drive / 'state/adaptation_state.json'
hpo_ckpt = hpo_drive / 'checkpoints/caduceus_hpo'
hpo_log = hpo_drive / 'logs/caduceus_hpo.log'
hpo_pidfile = hpo_drive / 'state/caduceus_hpo.pid'
assert hpo_root.exists() and hpo_drive.exists()
hpo_git = subprocess.run(['git','-C',str(hpo_root),'rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
hpo_branch = subprocess.run(['git','-C',str(hpo_root),'branch','--show-current'],capture_output=True,text=True,check=True).stdout.strip()
hpo_dirty = subprocess.run(['git','-C',str(hpo_root),'status','--porcelain'],capture_output=True,text=True,check=True).stdout.strip()
assert hpo_branch == 'research/posthoc-foundation-adaptation' and hpo_git.startswith('69ad3e6') and not hpo_dirty, (hpo_branch,hpo_git,hpo_dirty)
hpo_reference = Path('/content/Homo_sapiens_assembly38.fasta')
assert hpo_reference.exists() and not hpo_lock.exists(), 'reference missing or selection already locked; inspect before proceeding'
hpo_procs = subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout
hpo_active = [line.strip() for line in hpo_procs.splitlines() if 'run_caduceus_hpo.py' in line and 'ps -eo' not in line]
if hpo_active:
    print('Existing HPO process:', hpo_active)
else:
    hpo_out.parent.mkdir(parents=True,exist_ok=True); hpo_ckpt.mkdir(parents=True,exist_ok=True); hpo_log.parent.mkdir(parents=True,exist_ok=True); hpo_pidfile.parent.mkdir(parents=True,exist_ok=True)
    hpo_cmd = ['/content/caduceus-env/bin/python',str(hpo_root/'scripts/adaptation/run_caduceus_hpo.py'),'--root',str(hpo_root),'--reference',str(hpo_reference),'--cache-dir',str(hpo_drive/'model_cache'),'--checkpoint-dir',str(hpo_ckpt),'--state',str(hpo_state),'--output',str(hpo_out),'--device','cuda','--trials','8','--microbatch-size','2','--max-length','8192','--seed','42']
    hpo_handle = open(hpo_log,'a',encoding='utf-8',buffering=1)
    hpo_proc = subprocess.Popen(hpo_cmd,cwd=hpo_root,stdout=hpo_handle,stderr=subprocess.STDOUT,start_new_session=True)
    hpo_pidfile.write_text(str(hpo_proc.pid)+'\n',encoding='utf-8')
    hpo_handle.close()
    time.sleep(2)
    print('HPO pid=',hpo_proc.pid,'running=',hpo_proc.poll() is None,'log=',hpo_log,'resume_db_exists=',hpo_db.exists())
print('git=',hpo_git,'branch=',hpo_branch,'dirty=',bool(hpo_dirty),'active_count=',len(hpo_active),'selection_lock_exists=',hpo_lock.exists(),'train rows=3199 only; no validation or locked rows loaded')


HPO pid= 57847 running= False log= /content/drive/MyDrive/EvoVariantTR/logs/caduceus_hpo.log resume_db_exists= False
git= 69ad3e60174f6f5e902a09180503ab8e23ba2fb1 branch= research/posthoc-foundation-adaptation dirty= False active_count= 0 selection_lock_exists= False train rows=3199 only; no validation or locked rows loaded


In [ ]:
from pathlib import Path
hpo_log = Path('/content/drive/MyDrive/EvoVariantTR/logs/caduceus_hpo.log')
print(hpo_log.read_text(encoding='utf-8')[-12000:] if hpo_log.exists() else 'missing log')
print('pid file:',Path('/content/drive/MyDrive/EvoVariantTR/state/caduceus_hpo.pid').read_text() if Path('/content/drive/MyDrive/EvoVariantTR/state/caduceus_hpo.pid').exists() else 'missing')


Traceback (most recent call last):
  File "/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py", line 15, in <module>
    from evovariant_tr.adaptation.checkpointing import load_checkpoint, save_checkpoint
ModuleNotFoundError: No module named 'evovariant_tr'

pid file: 57847



In [ ]:
from pathlib import Path
import os, subprocess, time
hpo_root = Path('/content/EvoVariant')
hpo_drive = Path('/content/drive/MyDrive/EvoVariantTR')
hpo_out = hpo_drive/'hpo/caduceus_hpo.json'
hpo_lock = hpo_out.with_name('selection_closed.json')
hpo_reference = Path('/content/Homo_sapiens_assembly38.fasta')
hpo_log = hpo_drive/'logs/caduceus_hpo.log'
hpo_pidfile = hpo_drive/'state/caduceus_hpo.pid'
hpo_procs = subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout
hpo_active = [x.strip() for x in hpo_procs.splitlines() if 'run_caduceus_hpo.py' in x and 'ps -eo' not in x]
assert not hpo_active and not hpo_lock.exists() and hpo_reference.exists(), (hpo_active,hpo_lock.exists(),hpo_reference.exists())
hpo_log.parent.mkdir(parents=True,exist_ok=True); hpo_pidfile.parent.mkdir(parents=True,exist_ok=True)
hpo_cmd = ['/content/caduceus-env/bin/python',str(hpo_root/'scripts/adaptation/run_caduceus_hpo.py'),'--root',str(hpo_root),'--reference',str(hpo_reference),'--cache-dir',str(hpo_drive/'model_cache'),'--checkpoint-dir',str(hpo_drive/'checkpoints/caduceus_hpo'),'--state',str(hpo_drive/'state/adaptation_state.json'),'--output',str(hpo_out),'--device','cuda','--trials','8','--microbatch-size','2','--max-length','8192','--seed','42']
hpo_env = os.environ.copy(); hpo_env['PYTHONPATH'] = str(hpo_root/'src') + (os.pathsep+hpo_env['PYTHONPATH'] if hpo_env.get('PYTHONPATH') else '')
hpo_handle = open(hpo_log,'a',encoding='utf-8',buffering=1); hpo_handle.write('\n=== HPO start/resume with repository src on PYTHONPATH ===\n')
hpo_proc = subprocess.Popen(hpo_cmd,cwd=hpo_root,env=hpo_env,stdout=hpo_handle,stderr=subprocess.STDOUT,start_new_session=True)
hpo_pidfile.write_text(str(hpo_proc.pid)+'\n',encoding='utf-8'); hpo_handle.close(); time.sleep(3)
print('pid=',hpo_proc.pid,'running=',hpo_proc.poll() is None,'database=',hpo_out.with_suffix('.sqlite3').exists(),'log=',hpo_log)


pid= 58018 running= True database= False log= /content/drive/MyDrive/EvoVariantTR/logs/caduceus_hpo.log


In [ ]:
from pathlib import Path
import subprocess, time
r=Path('/content/drive/MyDrive/EvoVariantTR')
log=r/'logs/caduceus_hpo.log'
print('worker:')
print(subprocess.run(['ps','-eo','pid=,etime=,stat=,args='],capture_output=True,text=True).stdout[-4000:])
print('log tail:')
print(log.read_text(encoding='utf-8')[-5000:] if log.exists() else 'missing')
for folder in ('runs','state','hpo','checkpoints/caduceus_hpo','checkpoints/caduceus_frozen_head'):
 p=r/folder
 print('\n',folder,[(x.name,x.stat().st_size) for x in sorted(p.iterdir())] if p.exists() else 'missing')


worker:
2 --target_host=172.28.0.12 --tunnel_background_save_url=https://colab.research.google.com/tun/m/cc48301118ce562b961b3c22d803539adc1e0c19/gpu-t4-s-kkb-ass1a1-3tlp1cxc7bjmc --tunnel_background_save_delay=10s --tunnel_periodic_background_save_frequency=30m0s --enable_output_coalescing=true --output_coalescing_required=true --use_oneplatform_for_bg_save=true
     40    03:41:25 Ss   tail -n +0 -F /root/.config/Google/DriveFS/Logs/drive_fs.txt
     46    03:41:25 Ss   tail -n +0 -F /root/.config/Google/DriveFS/Logs/dpb.txt
     65    03:41:23 Z    [python3] <defunct>
     66    03:41:23 S    python3 /usr/local/bin/colab-fileshim.py
    109    03:41:22 Ss   sshd: /usr/sbin/sshd [listener] 0 of 1-1 startups
    124    03:41:21 Sl   /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerAp

In [ ]:
from pathlib import Path
import json, hashlib, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR')
run=json.loads((r/'checkpoints/caduceus_frozen_head/run.json').read_text())
print('baseline run keys=',sorted(run)); print(json.dumps(run,indent=2)[:6000])
st=json.loads((r/'state/adaptation_state.json').read_text()); print('state keys=',list(st)); print('last=',json.dumps(st,indent=2)[-2200:])
print('HPO summary exists=',(r/'hpo/caduceus_hpo.json').exists())
print('worker=',subprocess.run(['ps','-p','58018','-o','etime=,stat=,args='],capture_output=True,text=True).stdout.strip())


baseline run keys= ['checkpoint', 'checkpoint_bytes', 'checkpoint_sha256', 'config', 'config_sha256', 'data', 'epochs', 'holdout_evaluated', 'model_id', 'peak_vram_bytes', 'protocol_sha256', 'revision', 'runtime_seconds', 'seed', 'selection_scope', 'stage', 'status', 'total_parameters', 'trainable_parameters']
{
  "checkpoint": "/content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_frozen_head/latest.pt",
  "checkpoint_bytes": 31077754,
  "checkpoint_sha256": "230fc7bef5c12fcb5f5c5a6b31ff4d79d57c244b8a00d70ac54e60aaf4e02213",
  "config": {
    "batch_size": 1,
    "dropout": 0.1,
    "effective_batch_size": 16,
    "epochs": 3,
    "learning_rate": 2e-05,
    "max_length": 8192,
    "reference_sha256": "5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51",
    "seed": 42,
    "stage": "frozen_head_only",
    "weight_decay": 0.01
  },
  "config_sha256": "c6d515f0bb3188d5bdf203a36e53cddb14c7292e30c6d8cc6a1399b7b8396da1",
  "data": {
    "reference_sha256": "5be01555d98347

In [ ]:
from pathlib import Path
import subprocess, json, os, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); log=r/'logs/caduceus_hpo.log'
print(subprocess.run(['ps','-p','58018','-o','etime=,stat=,rss=,args='],capture_output=True,text=True).stdout)
print('log tail\n',log.read_text(encoding='utf-8')[-2500:])
print('gpu\n',subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout[-1600:])
print('checkpoint files',[(str(p.relative_to(r)),p.stat().st_size) for p in (r/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()][:30])
print('hpo output', (r/'hpo/caduceus_hpo.json').exists())


      03:37 Rsl  1347632 /content/caduceus-env/bin/python /content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py --root /content/EvoVariant --reference /content/Homo_sapiens_assembly38.fasta --cache-dir /content/drive/MyDrive/EvoVariantTR/model_cache --checkpoint-dir /content/drive/MyDrive/EvoVariantTR/checkpoints/caduceus_hpo --state /content/drive/MyDrive/EvoVariantTR/state/adaptation_state.json --output /content/drive/MyDrive/EvoVariantTR/hpo/caduceus_hpo.json --device cuda --trials 8 --microbatch-size 2 --max-length 8192 --seed 42

log tail
 Traceback (most recent call last):
  File "/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py", line 15, in <module>
    from evovariant_tr.adaptation.checkpointing import load_checkpoint, save_checkpoint
ModuleNotFoundError: No module named 'evovariant_tr'

=== HPO start/resume with repository src on PYTHONPATH ===
/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py:214: FutureWarning: RetryFailedTrialCallback has been depre

In [ ]:
from pathlib import Path
import subprocess, json, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); log=r/'logs/caduceus_hpo.log'
print(subprocess.run(['ps','-p','58018','-o','etime=,stat=,rss='],capture_output=True,text=True).stdout.strip())
print(log.read_text(encoding='utf-8')[-3200:])
print('fold artifacts:',[(str(p.relative_to(r)),p.stat().st_size) for p in (r/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()])
print('trial summary exists:',(r/'hpo/caduceus_hpo.json').exists())
print('gpu:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


05:00 Rsl  1347640
Traceback (most recent call last):
  File "/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py", line 15, in <module>
    from evovariant_tr.adaptation.checkpointing import load_checkpoint, save_checkpoint
ModuleNotFoundError: No module named 'evovariant_tr'

=== HPO start/resume with repository src on PYTHONPATH ===
/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py:214: FutureWarning: RetryFailedTrialCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `RetryHeartbeatStaleTrialCallback` instead.
  failed_trial_callback=optuna.storages.RetryFailedTrialCallback(max_retry=1),
/content/caduceus-env/lib/python3.11/site-packages/optuna/_deprecated.py:180: ExperimentalWarning: RetryHeartbeatStaleTrialCallback is experimental (supported from v2.8.0). The interface can change in the future.
  _original_init(self, *args, **kwargs)
/content/caduceus-env/lib/python3.11/site-

In [ ]:
from pathlib import Path
import subprocess, json, sqlite3, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); log=r/'logs/caduceus_hpo.log'; db=r/'hpo/caduceus_hpo.sqlite3'
print('process:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,rss='],capture_output=True,text=True).stdout.strip())
print('log tail:',log.read_text(encoding='utf-8')[-1600:])
print('checkpoint files:',[(p.name,p.stat().st_size) for p in (r/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()])
if db.exists():
 con=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',con.execute('select number,state from trials').fetchall()); con.close()


process: 07:42 Rsl  1394500
log tail: HPO start/resume with repository src on PYTHONPATH ===
/content/EvoVariant/scripts/adaptation/run_caduceus_hpo.py:214: FutureWarning: RetryFailedTrialCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `RetryHeartbeatStaleTrialCallback` instead.
  failed_trial_callback=optuna.storages.RetryFailedTrialCallback(max_retry=1),
/content/caduceus-env/lib/python3.11/site-packages/optuna/_deprecated.py:180: ExperimentalWarning: RetryHeartbeatStaleTrialCallback is experimental (supported from v2.8.0). The interface can change in the future.
  _original_init(self, *args, **kwargs)
/content/caduceus-env/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``heartbeat_interval`` is an experimental feature. The interface can change in the future.
  optuna_warn(
/content/caduceus-env/lib/python3.11/site-packages/optuna/storages/_rdb/storage

In [ ]:
from pathlib import Path
import sqlite3, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); db=r/'hpo/caduceus_hpo.sqlite3'; log=r/'logs/caduceus_hpo.log'
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,rss='],capture_output=True,text=True).stdout.strip())
print('trial folds:',[(str(p.relative_to(r)),p.stat().st_size) for p in (r/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()])
con=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trial states:',con.execute('select number,state from trials').fetchall()); con.close()
print('log:',log.read_text(encoding='utf-8')[-1000:])
print('GPU:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


worker: 13:22 Rsl  1394512
trial folds: []
trial states: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]
log: e can change in the future.
  _original_init(self, *args, **kwargs)
/content/caduceus-env/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``heartbeat_interval`` is an experimental feature. The interface can change in the future.
  optuna_warn(
/content/caduceus-env/lib/python3.11/site-packages/optuna/storages/_rdb/storage.py:238: FutureWarning: `failed_trial_callback` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `heartbeat_stale_trial_callback` instead.
  optuna_warn(
[I 2026-09-22 20:46:29,666] A new study created in RDB with name: evovariant_caduceus_train_grouped_cv_v1
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 

In [ ]:
from pathlib import Path
import sqlite3, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); db=r/'hpo/caduceus_hpo.sqlite3'; log=r/'logs/caduceus_hpo.log'
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,rss='],capture_output=True,text=True).stdout.strip())
print('artifacts:',[(str(p.relative_to(r)),p.stat().st_size) for p in (r/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()])
con=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',con.execute('select number,state from trials').fetchall()); con.close()
print('GPU:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())
print('log tail:',log.read_text(encoding='utf-8')[-500:])


worker: 01:29:06 Rsl  1427368
artifacts: [('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/best.pt', 31077162), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json', 2999), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/latest.pt', 31078330), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/best.pt', 31077162), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/history.json', 702), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/latest.pt', 31078330)]
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]
GPU: 100 %, 921 MiB, 15360 MiB
log tail: d=True`.
  warnings.warn(
/content/caduceus-env/lib/python3.11/site-packages/optuna/trial/_trial.py:505: UserWarning: The reported value is ignored because this `step=0` is already reported.
  optuna_warn(
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume whe

In [ ]:
from pathlib import Path
import sqlite3
p=Path('/content/drive/MyDrive/EvoVariantTR/hpo/caduceus_hpo.sqlite3')
c=sqlite3.connect(f'file:{p}?mode=ro',uri=True)
print(c.execute('select t.number,p.param_name,p.param_value,p.distribution_json from trials t join trial_params p on p.trial_id=t.trial_id where t.number=0').fetchall())
c.close()


[(0, 'learning_rate', 1.0, '{"name": "CategoricalDistribution", "attributes": {"choices": [1e-05, 3e-05, 0.0001]}}'), (0, 'weight_decay', 0.0, '{"name": "CategoricalDistribution", "attributes": {"choices": [0.0, 0.01, 0.05]}}'), (0, 'dropout', 1.0, '{"name": "CategoricalDistribution", "attributes": {"choices": [0.0, 0.1, 0.2]}}'), (0, 'regime', 0.0, '{"name": "CategoricalDistribution", "attributes": {"choices": ["frozen_head_only", "partial_small", "partial_large", "full_if_feasible"]}}'), (0, 'effective_batch_size', 0.0, '{"name": "CategoricalDistribution", "attributes": {"choices": [16, 32]}}'), (0, 'epoch_cap', 0.0, '{"name": "CategoricalDistribution", "attributes": {"choices": [4, 6, 8]}}')]


In [ ]:
import subprocess
root=Path('/content/EvoVariant')
pull=subprocess.run(['git','-C',str(root),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],capture_output=True,text=True,check=True)
head=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
branch=subprocess.check_output(['git','-C',str(root),'branch','--show-current'],text=True).strip()
dirty=subprocess.check_output(['git','-C',str(root),'status','--porcelain'],text=True).strip()
assert branch=='research/posthoc-foundation-adaptation' and head.startswith('a4c6cd0') and not dirty,(branch,head,dirty)
print(pull.stdout+pull.stderr); print('branch=',branch,'HEAD=',head,'dirty=',bool(dirty))
print('OOF calibration runner=',(root/'scripts/adaptation/run_posthoc_analysis.py').exists(),'calibrated evaluator=', '--calibration-report' in (root/'scripts/adaptation/evaluate_caduceus.py').read_text())


Updating 69ad3e6..a4c6cd0
Fast-forward
 README.md                                          |  9 ++-
 docs/CONTRIBUTION_MAP.md                           |  4 +-
 docs/JUDGE_DEMO.md                                 |  2 +-
 .../EvoVariant_TR_Adaptation_Autonomous.ipynb      | 24 +++++-
 research/adaptation/DECISIONS.md                   |  2 +-
 research/adaptation/EXPERIMENT_LEDGER.md           |  6 +-
 research/adaptation/STATE.md                       |  9 ++-
 .../reports/adaptation/FINAL_ADAPTATION_REPORT.md  | 10 +--
 scripts/adaptation/evaluate_caduceus.py            | 92 +++++++++++++++++++---
 src/evovariant_tr/adaptation/calibration.py        | 38 +++++++++
 tests/adaptation/test_research_helpers.py          | 14 ++++
 11 files changed, 176 insertions(+), 34 deletions(-)
From https://github.com/UtkarsHMer05/EvoVariant-TR-
 * branch            research/posthoc-foundation-adaptation -> FETCH_HEAD
   69ad3e6..a4c6cd0  research/posthoc-foundation-adaptation -> origin/research/postho

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, time
finish_root=Path('/content/EvoVariant')
finish_drive=Path('/content/drive/MyDrive/EvoVariantTR')
finish_py='/content/caduceus-env/bin/python'
finish_reference=Path('/content/Homo_sapiens_assembly38.fasta')
finish_lock=finish_drive/'hpo/selection_closed.json'
finish_state=finish_drive/'state/adaptation_state.json'
finish_calibration=finish_drive/'analysis/caduceus_train_oof_calibration.json'
finish_run=finish_drive/'runs/caduceus_final'
finish_report=finish_run/'run.json'
finish_checkpoint=finish_run/'latest.pt'
if not finish_lock.exists():
    print('Blocked: HPO selection is open; no final refit or holdout access.')
else:
    finish_selection=json.loads(finish_lock.read_text())
    finish_oof=Path(finish_selection['train_oof_csv'])
    finish_calibration.parent.mkdir(parents=True,exist_ok=True)
    if not finish_oof.exists():
        print('Blocked: selected TRAIN OOF file is missing.')
    else:
        if not finish_calibration.exists():
            subprocess.run([finish_py,str(finish_root/'scripts/adaptation/run_posthoc_analysis.py'),'--root',str(finish_root),'--predictions',str(finish_oof),'--output',str(finish_calibration),'--state',str(finish_state)],cwd=finish_root,check=True,env={**os.environ,'PYTHONPATH':str(finish_root/'src')})
        finish_cal=json.loads(finish_calibration.read_text())
        finish_match=(finish_cal.get('fit_scope')=='TRAIN_OOF_ONLY' and finish_cal.get('holdout_used_for_fit') is False and finish_cal.get('protocol_sha256')==finish_selection.get('protocol_sha256') and finish_cal.get('oof_predictions_sha256')==finish_selection.get('train_oof_sha256'))
        finish_processes=subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout.splitlines()
        finish_hpo_active=any('run_caduceus_hpo.py' in line for line in finish_processes)
        finish_final_active=any('train_caduceus.py' in line for line in finish_processes)
        if not finish_match:
            print('Blocked: calibration report does not match selected TRAIN OOF evidence.')
        elif finish_hpo_active:
            print('Calibration is TRAIN-OOF-only and verified. Waiting for the HPO worker to exit before final refit.')
        elif finish_report.exists():
            finish_saved=json.loads(finish_report.read_text())
            finish_hash=hashlib.sha256(finish_lock.read_bytes()).hexdigest()
            print('Final run exists; selection lock matches=',finish_saved.get('config',{}).get('selection_lock_sha256')==finish_hash,'checkpoint exists=',finish_checkpoint.exists())
        elif finish_final_active:
            print('Final refit process is already active.')
        else:
            finish_params=finish_selection['selected_params']
            finish_cmd=[finish_py,str(finish_root/'scripts/adaptation/train_caduceus.py'),'--root',str(finish_root),'--reference',str(finish_reference),'--cache-dir',str(finish_drive/'model_cache'),'--output-dir',str(finish_run),'--state',str(finish_state),'--selection-lock',str(finish_lock),'--device','cuda','--stage',finish_params['regime'],'--epochs',str(finish_selection['final_epochs']),'--batch-size','1','--effective-batch-size',str(finish_params['effective_batch_size']),'--dropout',str(finish_params['dropout']),'--max-length','8192','--learning-rate',str(finish_params['learning_rate']),'--weight-decay',str(finish_params['weight_decay']),'--seed',str(finish_selection['seed'])]
            if finish_checkpoint.exists(): finish_cmd += ['--resume',str(finish_checkpoint)]
            finish_log=finish_drive/'logs/caduceus_final.log'; finish_log.parent.mkdir(parents=True,exist_ok=True)
            finish_handle=open(finish_log,'a',encoding='utf-8',buffering=1)
            finish_env=os.environ.copy(); finish_env['PYTHONPATH']=str(finish_root/'src')+(os.pathsep+finish_env['PYTHONPATH'] if finish_env.get('PYTHONPATH') else '')
            finish_proc=subprocess.Popen(finish_cmd,cwd=finish_root,env=finish_env,stdout=finish_handle,stderr=subprocess.STDOUT,start_new_session=True)
            (finish_drive/'state/caduceus_final.pid').write_text(str(finish_proc.pid)+'\n'); finish_handle.close(); time.sleep(2)
            print('Started final TRAIN-only refit PID',finish_proc.pid,'running=',finish_proc.poll() is None,'log=',finish_log)


Blocked: HPO selection is open; no final refit or holdout access.


In [ ]:
from pathlib import Path
import json, os, subprocess

root=Path('/content/EvoVariant'); drive=Path('/content/drive/MyDrive/EvoVariantTR')
lock=drive/'hpo/selection_closed.json'; primary=drive/'runs/caduceus_final/run.json'
if not lock.exists():
    print('Blocked: HPO selection is still open; no robustness refits or holdout access.')
else:
    selection=json.loads(lock.read_text()); lock_hash=__import__('hashlib').sha256(lock.read_bytes()).hexdigest()
    if selection.get('status')!='SELECTION_CLOSED' or selection.get('holdout_evaluated') is not False or selection.get('seed')!=42:
        raise RuntimeError('selection lock does not match the frozen seed-42 protocol')
    if not primary.exists():
        print('Blocked: primary seed-42 final TRAIN refit is incomplete.')
    else:
        primary_run=json.loads(primary.read_text())
        if primary_run.get('status')!='PASS' or primary_run.get('seed')!=42 or primary_run.get('config',{}).get('selection_lock_sha256')!=lock_hash:
            raise RuntimeError('primary refit does not match the closed selection lock')
        env={**os.environ,'PYTHONPATH':str(root/'src')}; params=selection['selected_params']
        for seed in (1337,2026):
            out=drive/f'runs/caduceus_final_seed{seed}'; report_path=out/'run.json'; checkpoint=out/'latest.pt'
            if report_path.exists():
                report=json.loads(report_path.read_text())
                if report.get('status')!='PASS' or report.get('seed')!=seed or report.get('seed_mode')!='fixed_seed_robustness' or report.get('config',{}).get('selection_lock_sha256')!=lock_hash or not checkpoint.exists():
                    raise RuntimeError(f'seed-{seed} run exists but fails provenance checks')
                print(f'seed {seed}: verified existing TRAIN-only run')
                continue
            processes=subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout.splitlines()
            if any('run_caduceus_hpo.py' in line for line in processes):
                print('Blocked: HPO worker is still active; final refits wait.')
                break
            if any('train_caduceus.py' in line and f'--seed {seed}' in line for line in processes):
                print(f'seed {seed} training is already active; wait for its checkpointed run.json.')
                break
            command=[str(root.parent/'caduceus-env/bin/python'),str(root/'scripts/adaptation/train_caduceus.py'),'--root',str(root),'--reference','/content/Homo_sapiens_assembly38.fasta','--cache-dir',str(drive/'model_cache'),'--output-dir',str(out),'--state',str(drive/'state/adaptation_state.json'),'--selection-lock',str(lock),'--device','cuda','--stage',params['regime'],'--epochs',str(selection['final_epochs']),'--batch-size','1','--effective-batch-size',str(params['effective_batch_size']),'--dropout',str(params['dropout']),'--max-length','8192','--learning-rate',str(params['learning_rate']),'--weight-decay',str(params['weight_decay']),'--seed',str(seed),'--fixed-seed-robustness']
            if checkpoint.exists(): command.extend(['--resume',str(checkpoint)])
            log_path=drive/f'logs/caduceus_final_seed{seed}.log'; log_path.parent.mkdir(parents=True,exist_ok=True)
            print(f'starting seed {seed} TRAIN-only refit; log={log_path}')
            with log_path.open('a',encoding='utf-8',buffering=1) as log:
                subprocess.run(command,cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT,check=True)
            report=json.loads(report_path.read_text())
            if report.get('status')!='PASS' or report.get('seed')!=seed or report.get('seed_mode')!='fixed_seed_robustness' or report.get('config',{}).get('selection_lock_sha256')!=lock_hash:
                raise RuntimeError(f'seed-{seed} TRAIN-only refit failed verification')
            print(f'seed {seed}: TRAIN-only refit PASS')


Blocked: HPO selection is still open; no robustness refits or holdout access.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess

final_root=Path('/content/EvoVariant'); final_drive=Path('/content/drive/MyDrive/EvoVariantTR')
final_py='/content/caduceus-env/bin/python'; final_reference=Path('/content/Homo_sapiens_assembly38.fasta')
final_lock=final_drive/'hpo/selection_closed.json'; final_calibration=final_drive/'analysis/caduceus_train_oof_calibration.json'
seeds=(42,1337,2026)
if not final_lock.exists():
    print('Blocked: HPO selection is open; VALIDATION remains closed.')
elif not final_calibration.exists():
    print('Blocked: TRAIN-OOF calibration report is missing.')
else:
    final_selection=json.loads(final_lock.read_text()); final_cal=json.loads(final_calibration.read_text())
    final_lock_hash=hashlib.sha256(final_lock.read_bytes()).hexdigest()
    final_cal_hash=hashlib.sha256(final_calibration.read_bytes()).hexdigest()
    if final_selection.get('status')!='SELECTION_CLOSED' or final_selection.get('holdout_evaluated') is not False or final_selection.get('seed')!=42 or final_cal.get('fit_scope')!='TRAIN_OOF_ONLY' or final_cal.get('holdout_used_for_fit') is not False or final_cal.get('protocol_sha256')!=final_selection.get('protocol_sha256') or final_cal.get('oof_predictions_sha256')!=final_selection.get('train_oof_sha256'):
        raise RuntimeError('selection lock or calibration report failed TRAIN-only provenance checks')
    run_dirs={42:final_drive/'runs/caduceus_final',1337:final_drive/'runs/caduceus_final_seed1337',2026:final_drive/'runs/caduceus_final_seed2026'}
    run_reports={seed:json.loads((run_dirs[seed]/'run.json').read_text()) if (run_dirs[seed]/'run.json').exists() else None for seed in seeds}
    missing=[seed for seed in seeds if run_reports[seed] is None or not (run_dirs[seed]/'latest.pt').exists() or run_reports[seed].get('status')!='PASS' or run_reports[seed].get('seed')!=seed or run_reports[seed].get('holdout_evaluated') is not False or run_reports[seed].get('config',{}).get('selection_lock_sha256')!=final_lock_hash]
    if missing:
        print('Blocked: verified TRAIN-only final checkpoints are missing for seeds',missing,'; no holdout access.')
    else:
        final_processes=subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout.splitlines()
        if any('run_caduceus_hpo.py' in line or 'train_caduceus.py' in line for line in final_processes):
            print('Blocked: HPO/final TRAIN worker is still active; all holdout systems remain closed.')
        else:
            for seed in seeds:
                suffix='' if seed==42 else f'_seed{seed}'
                checkpoint=run_dirs[seed]/'latest.pt'
                output=final_drive/f'exports/caduceus_validation{suffix}.csv'
                report_path=output.with_suffix('.json'); attempt_path=output.with_name(f'{output.stem}.attempt.json')
                existing=[path.exists() for path in (output,report_path,attempt_path)]
                if any(existing):
                    if not all(existing):
                        raise RuntimeError(f'seed-{seed} has a partial one-shot evaluation; do not rerun')
                    report=json.loads(report_path.read_text())
                    if report.get('status')!='PASS' or report.get('rows')!=801 or report.get('cohort_label')!='POST-HOC ADAPTATION HOLDOUT' or report.get('seed_mode')!=('primary' if seed==42 else 'fixed_seed_robustness') or report.get('selection_lock_sha256')!=final_lock_hash or report.get('calibration_report_sha256')!=final_cal_hash or report.get('evaluation_attempt_sha256')!=hashlib.sha256(attempt_path.read_bytes()).hexdigest():
                        raise RuntimeError(f'seed-{seed} existing holdout report failed verification')
                    print(f'seed {seed}: verified completed one-shot holdout')
                    continue
                command=[final_py,str(final_root/'scripts/adaptation/evaluate_caduceus.py'),'--root',str(final_root),'--reference',str(final_reference),'--cache-dir',str(final_drive/'model_cache'),'--checkpoint',str(checkpoint),'--selection-lock',str(final_lock),'--calibration-report',str(final_calibration),'--state',str(final_drive/'state/adaptation_state.json'),'--output',str(output),'--split','validation','--device','cuda','--seed',str(seed)]
                if seed!=42: command.append('--fixed-seed-robustness')
                final_env={**os.environ,'PYTHONPATH':str(final_root/'src')+ (os.pathsep+os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')}
                subprocess.run(command,cwd=final_root,env=final_env,check=True)
                report=json.loads(report_path.read_text())
                if report.get('status')!='PASS' or report.get('rows')!=801 or report.get('selection_lock_sha256')!=final_lock_hash or report.get('calibration_report_sha256')!=final_cal_hash:
                    raise RuntimeError(f'seed-{seed} one-shot holdout failed post-run verification')
                print(f'seed {seed}: one-shot holdout PASS')

Blocked: HPO selection is open; VALIDATION remains closed.


In [ ]:
from pathlib import Path
import sqlite3, subprocess
hpo_drive=Path('/content/drive/MyDrive/EvoVariantTR'); hpo_db=hpo_drive/'hpo/caduceus_hpo.sqlite3'; hpo_log=hpo_drive/'logs/caduceus_hpo.log'
print(subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu=,rss='],capture_output=True,text=True).stdout.strip())
print('fold checkpoint files:',[(str(p.relative_to(hpo_drive)),p.stat().st_size) for p in (hpo_drive/'checkpoints/caduceus_hpo').rglob('*') if p.is_file()])
con=sqlite3.connect(f'file:{hpo_db}?mode=ro',uri=True); print('trials:',con.execute('select number,state from trials').fetchall()); con.close()
print('GPU:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())
print('log tail:',hpo_log.read_text(encoding='utf-8')[-500:])


02:01:00 Rsl  98.3 1427368
fold checkpoint files: [('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/best.pt', 31077162), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json', 2999), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/latest.pt', 31078330), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/best.pt', 31077162), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/history.json', 2101), ('checkpoints/caduceus_hpo/705d3dee50f29527/fold_1/latest.pt', 31078330)]
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]
GPU: 69 %, 947 MiB, 15360 MiB, 72
log tail:  reported.
  optuna_warn(
/content/caduceus-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/content/caduceus-env/lib/python3.11/site-packages/optuna/trial/_trial.py:

In [ ]:
from pathlib import Path
import json, sqlite3, subprocess, time
r=Path('/content/drive/MyDrive/EvoVariantTR')
history=next((r/'checkpoints/caduceus_hpo').rglob('history.json'))
print('first fold history:',json.dumps(json.loads(history.read_text()),indent=2))
print('checkpoint mtime UTC:',time.strftime('%Y-%m-%d %H:%M:%S',time.gmtime(history.stat().st_mtime)))
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())


first fold history: [
  {
    "epoch": 1,
    "learning_rate": 3e-05,
    "peak_vram_bytes": 607967232,
    "train_loss": 1.174378810496717,
    "validation": {
      "accuracy": 0.17354596622889307,
      "auprc": 0.20407563220689193,
      "auroc": 0.5399791391845875,
      "balanced_accuracy": 0.5,
      "brier": 0.28881228900556777,
      "confusion_matrix": {
        "fn": 0,
        "fp": 881,
        "tn": 0,
        "tp": 185
      },
      "ece": 0.38145234789454685,
      "f1": 0.29576338928856916,
      "log_loss": 0.7711006571778737,
      "mcc": 0.0,
      "n": 1066,
      "negatives": 881,
      "positives": 185,
      "precision": 0.17354596622889307,
      "recall": 1.0,
      "specificity": 0.0,
      "threshold": 0.5
    }
  }
]
checkpoint mtime UTC: 2026-09-22 21:02:32
worker: 18:32 Rsl  97.1


In [ ]:
from pathlib import Path
import json, subprocess, sqlite3, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for path in sorted(root.rglob('history.json')):
 print(path.relative_to(r),json.loads(path.read_text()))
print('process:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu=,rss='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; con=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',con.execute('select number,state from trials').fetchall()); con.close()


process: 


OperationalError: unable to open database file

In [ ]:
from pathlib import Path
import json, sqlite3, subprocess, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),json.loads(p.read_text()))
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()
print('GPU:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,temperature.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [{'epoch': 1, 'learning_rate': 3e-05, 'peak_vram_bytes': 607967232, 'train_loss': 1.174378810496717, 'validation': {'accuracy': 0.17354596622889307, 'auprc': 0.20407563220689193, 'auroc': 0.5399791391845875, 'balanced_accuracy': 0.5, 'brier': 0.28881228900556777, 'confusion_matrix': {'fn': 0, 'fp': 881, 'tn': 0, 'tp': 185}, 'ece': 0.38145234789454685, 'f1': 0.29576338928856916, 'log_loss': 0.7711006571778737, 'mcc': 0.0, 'n': 1066, 'negatives': 881, 'positives': 185, 'precision': 0.17354596622889307, 'recall': 1.0, 'specificity': 0.0, 'threshold': 0.5}}]
worker: 22:31 Rsl  97.3
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]
GPU: 75 %, 867 MiB, 71


In [ ]:
from pathlib import Path
import json, sqlite3, subprocess, time
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),json.loads(p.read_text()))
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [{'epoch': 1, 'learning_rate': 3e-05, 'peak_vram_bytes': 607967232, 'train_loss': 1.174378810496717, 'validation': {'accuracy': 0.17354596622889307, 'auprc': 0.20407563220689193, 'auroc': 0.5399791391845875, 'balanced_accuracy': 0.5, 'brier': 0.28881228900556777, 'confusion_matrix': {'fn': 0, 'fp': 881, 'tn': 0, 'tp': 185}, 'ece': 0.38145234789454685, 'f1': 0.29576338928856916, 'log_loss': 0.7711006571778737, 'mcc': 0.0, 'n': 1066, 'negatives': 881, 'positives': 185, 'precision': 0.17354596622889307, 'recall': 1.0, 'specificity': 0.0, 'threshold': 0.5}}]
worker: 26:31 Rsl  97.4
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
root=Path('/content/EvoVariant')
pull=subprocess.run(['git','-C',str(root),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'],capture_output=True,text=True,check=True)
head=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
status=subprocess.check_output(['git','-C',str(root),'status','--porcelain'],text=True).strip()
assert head.startswith('46047ae') and not status,(head,status)
print(pull.stdout+pull.stderr); print('HEAD=',head,'dirty=',bool(status),'attempt-marker guard=',"ATTEMPT_STARTED" in (root/'scripts/adaptation/evaluate_caduceus.py').read_text())


Updating a4c6cd0..46047ae
Fast-forward
 .../EvoVariant_TR_Adaptation_Autonomous.ipynb      |  3 +-
 research/adaptation/EXPERIMENT_LEDGER.md           |  2 +-
 research/adaptation/STATE.md                       |  2 +-
 scripts/adaptation/evaluate_caduceus.py            | 45 ++++++++++++++++++----
 4 files changed, 42 insertions(+), 10 deletions(-)
From https://github.com/UtkarsHMer05/EvoVariant-TR-
 * branch            research/posthoc-foundation-adaptation -> FETCH_HEAD
   a4c6cd0..46047ae  research/posthoc-foundation-adaptation -> origin/research/posthoc-foundation-adaptation

HEAD= 46047aee2473ea2ea2de693945f45a16ba5b09cb dirty= False attempt-marker guard= True


In [ ]:
from pathlib import Path
import json, sqlite3, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),[(e['epoch'],e['train_loss'],e['validation']['auroc']) for e in json.loads(p.read_text())])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [(1, 1.174378810496717, 0.5399791391845875)]
worker: 27:38 Rsl  97.3
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
from pathlib import Path
import json, subprocess, sqlite3
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print([(e['epoch'],e['train_loss'],e['validation']['auroc']) for e in json.loads(p.read_text())])
print(subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print(c.execute('select number,state from trials').fetchall()); c.close()


[(1, 1.174378810496717, 0.5399791391845875)]
29:20 Rsl  97.4
[(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
from pathlib import Path
import json, sqlite3, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
print([(str(p.relative_to(r)),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())]) for p in sorted(root.rglob('history.json'))])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()
print('GPU:',subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,temperature.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


[('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json', [(1, 1.174379, 0.539979)])]
worker: 30:37 Rsl  97.4
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]
GPU: 100 %, 867 MiB, 72


In [ ]:
from pathlib import Path
import json, sqlite3, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
print([(str(p.relative_to(r)),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())]) for p in sorted(root.rglob('history.json'))])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()


[('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json', [(1, 1.174379, 0.539979)])]
worker: 31:43 Rsl  97.4
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
from pathlib import Path
import json, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); p=next((r/'checkpoints/caduceus_hpo').rglob('history.json'))
print('epochs:',[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())


epochs: [(1, 1.174379, 0.539979), (2, 1.159086, 0.541792)]
worker: 32:45 Rsl  97.4


In [ ]:
from pathlib import Path
import json, subprocess, sqlite3
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
print([(str(p.relative_to(r)),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())]) for p in sorted(root.rglob('history.json'))])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()


[('checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json', [(1, 1.174379, 0.539979), (2, 1.159086, 0.541792)])]
worker: 33:50 Rsl  97.5
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
from pathlib import Path
import json, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [(1, 1.174379, 0.539979), (2, 1.159086, 0.541792)]
worker: 35:40 Rsl  97.5


In [ ]:
from pathlib import Path
import json, subprocess, sqlite3
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())
db=r/'hpo/caduceus_hpo.sqlite3'; c=sqlite3.connect(f'file:{db}?mode=ro',uri=True); print('trials:',c.execute('select number,state from trials').fetchall()); c.close()


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [(1, 1.174379, 0.539979), (2, 1.159086, 0.541792), (3, 1.153331, 0.538878), (4, 1.146972, 0.538577)]
worker: 01:10:58 Rsl  98.0
trials: [(0, 'RUNNING'), (1, 'WAITING'), (2, 'WAITING'), (3, 'WAITING')]


In [ ]:
from pathlib import Path
import json, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print([(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())])
print(subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())


[(1, 1.174379, 0.539979), (2, 1.159086, 0.541792), (3, 1.153331, 0.538878), (4, 1.146972, 0.538577)]
[(1, 1.15206, 0.51209), (2, 1.149482, 0.512096)]
01:54:28 Rsl  98.3


In [ ]:
from pathlib import Path
import json, subprocess
r=Path('/content/drive/MyDrive/EvoVariantTR'); root=r/'checkpoints/caduceus_hpo'
for p in sorted(root.rglob('history.json')): print(p.relative_to(r),[(e['epoch'],round(e['train_loss'],6),round(e['validation']['auroc'],6)) for e in json.loads(p.read_text())])
print('worker:',subprocess.run(['ps','-p','58018','-o','etime=,stat=,pcpu='],capture_output=True,text=True).stdout.strip())


checkpoints/caduceus_hpo/705d3dee50f29527/fold_0/history.json [(1, 1.174379, 0.539979), (2, 1.159086, 0.541792), (3, 1.153331, 0.538878), (4, 1.146972, 0.538577)]
worker: 01:09:30 Rsl  97.9


In [ ]:
from pathlib import Path
import subprocess
root=Path('/content/EvoVariant')
head=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
branch=subprocess.check_output(['git','-C',str(root),'branch','--show-current'],text=True).strip()
dirty=subprocess.check_output(['git','-C',str(root),'status','--porcelain'],text=True).strip()
print({'branch':branch,'HEAD':head,'dirty':bool(dirty)})
assert branch=='research/posthoc-foundation-adaptation' and head.startswith('a207085') and not dirty


{'branch': 'research/posthoc-foundation-adaptation', 'HEAD': 'a207085ef33429470506e559aad5561f5d154130', 'dirty': False}
